# 1 - Config & Import

## 1.1 - Config

In [3]:
# 🔧 config import
import os

logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

[2025-06-28 11:34:51] INFO - 1353130262.py - Logger initialized (Notebook)


## 1.2 - Import

In [4]:
# 📦 External libs
import plotly.graph_objects as go
import importlib

# 📂 Internal modules
from Class import Data, Models, Trainer
import Class.Data.Features
import Class.Data.Labels
import Class.Data.Preprocessing
import Class.Data.DataAnalysis


def reload_modules():
    importlib.reload(Class.Data.DataFetcher)
    importlib.reload(Class.Data.Features)
    importlib.reload(Class.Data.Labels)
    importlib.reload(Class.Data.DataAnalysis)
    importlib.reload(Class.Data.Preprocessing)
    importlib.reload(Class.Models.LSTMModel)
    importlib.reload(Class.Trainer.LSTMTrainer)

[2025-06-28 11:35:00] INFO - DataFetcher.py - Logger initialized (DataFetcher.py)
[2025-06-28 11:35:00] INFO - Features.py - Logger initialized (Features.py)
[2025-06-28 11:35:00] INFO - Labels.py - Logger initialized (Labels.py)
[2025-06-28 11:35:00] INFO - Preprocessing.py - Logger initialized (Preprocessing.py)
[2025-06-28 11:35:00] INFO - LSTMModel.py - Logger initialized (LSTMModel.py)
[2025-06-28 11:35:02] INFO - DataAnalysis.py - Logger initialized (DataAnalysis.py)
[2025-06-28 11:35:02] INFO - LSTMTrainer.py - Logger initialized (LSTMTrainer.py)


# 2 - Data

## 2.1 - Data fetcher

### Variables

In [7]:
 #📊 data parameters
symbol = "EURUSD"
start_date = "12-06-2025"
end_date = "13-06-2025"
interval = "1s"
interval ="d"

### Process

In [6]:
from ib_insync import *
from datetime import datetime, timedelta
import pandas as pd
import asyncio
import random

ib = IB()

async def get_IB_FX_Data(symbol: str, start_date: datetime, end_date: datetime) -> pd.DataFrame:
    await ib.connectAsync('127.0.0.1', 4002, clientId=random.randint(100, 999))

    contract = Forex(symbol)
    data = []

    slice_duration = timedelta(minutes=1)
    total_requests = int((end_date - start_date) / slice_duration)
    print(f"Total requests to perform: {total_requests}")

    current_start = start_date

    for i in range(total_requests):
        current_end = min(current_start + slice_duration, end_date)

        try:
            ticks = await ib.reqHistoricalTicksAsync(
                contract,
                startDateTime=current_start,
                endDateTime=current_end,
                numberOfTicks=1000,
                whatToShow='BID_ASK',
                useRth=False
            )

            for tick in ticks:
                data.append({
                    'time': tick.time,
                    'bid': tick.priceBid,
                    'ask': tick.priceAsk,
                    'bidSize': tick.sizeBid,
                    'askSize': tick.sizeAsk
                })

        except Exception as e:
            print(f"Error fetching {symbol} from {current_start} to {current_end}: {e}")

        print(f"{i+1}/{total_requests}")
        current_start = current_end
        await asyncio.sleep(0.3)  # avoid pacing violation

    ib.disconnect()

    df = pd.DataFrame(data)
    return df


import nest_asyncio
nest_asyncio.apply()

start = datetime.utcnow() - timedelta(hours=1)
end = datetime.utcnow()

df = asyncio.run(get_IB_FX_Data(symbol, start, end))
print(df.head())


[2025-06-28 11:35:03] INFO - client.py - Connecting to 127.0.0.1:4002 with clientId 227...
[2025-06-28 11:35:05] INFO - client.py - Disconnecting
[2025-06-28 11:35:05] ERROR - client.py - API connection failed: ConnectionRefusedError(10061, "Connect call failed ('127.0.0.1', 4002)")
[2025-06-28 11:35:05] ERROR - client.py - Make sure API port on TWS/IBG is open


ConnectionRefusedError: [Errno 10061] Connect call failed ('127.0.0.1', 4002)

In [ ]:
reload_modules()

# 📥 Load data
fetcher = Class.Data.DataFetcher.DataFetcher(symbol, start_date, end_date, interval)

#fetcher.get_binance_data()

#fetcher.save_to_csv(directory="./data")

fetcher.load_from_csv_ib(directory="./data")

fetcher.resample_to_1m_ohlcv()

### Visualization

In [ ]:
print("#-----------------------")
print(fetcher.raw_data.columns)
print("#-----------------------")
print(fetcher.raw_data.head())
print("#-----------------------")
print(fetcher.raw_data.shape)
print("#-----------------------")

size = 200
df = fetcher.raw_data.copy()[-size:]

# Create a candlestick chart using Plotly
fig = go.Figure(data=[
    go.Candlestick(
        x=df.index,  # Using the timestamp index directly
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name="Candlesticks"
    )
])

# Customize layout
fig.update_layout(
    title=f"OHLC Candlestick Chart (Last {size} Points)",
    xaxis_title="Time",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=500,
    width=int(500 * 1.618)
)

# Show the chart
fig.show()

# Variables

# 1 - Get Raw Data From API

## 1.1 - Binance API

## 1.2 IB API

# 2 - Clear Raw Data

# 3 - Visualize Data

# 4 - Save Data to CSV